# 02. Exploratory Data Analysis & Sensor Degradation

In-depth statistical exploration covering multi-collinearity, hierarchical clustering dendrograms, grouped outlier boxplots, and event-centric degradation trajectories.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.cluster.hierarchy as sch
from scipy.spatial.distance import squareform
from src.data.load_data import load_workbook
from src.utils.io import load_config

sns.set_theme(style="whitegrid")
config = load_config()
workbook = load_workbook(config["project"]["raw_file"])
telemetry = workbook["telemetry"].sort_values(["Machine ID", "Timestamp"]).reset_index(drop=True)
events = workbook["events"]
numeric_cols = [c for c in telemetry.select_dtypes(include=np.number).columns if c != "SMR"]


## 1. Sensor Distributions Across Dozers


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, col in enumerate(numeric_cols):
    sns.kdeplot(data=telemetry, x=col, hue="Machine ID", common_norm=False, ax=axes[idx], fill=True, alpha=0.2)
    axes[idx].set_title(f"KDE Distribution: {col}", fontsize=12)
    axes[idx].grid(True)

plt.tight_layout()
plt.show()


## 2. Multi-Collinearity Analysis (|r| > 0.75)


In [ ]:
corr_matrix = telemetry[numeric_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".2f", linewidths=0.5)
plt.title("Sensor Pearson Correlation Heatmap", fontsize=14)
plt.tight_layout()
plt.show()


## 3. Hierarchical Feature Clustering Dendrogram


In [ ]:
distance_matrix = 1 - abs(corr_matrix.fillna(0))
condensed_dist = squareform(distance_matrix, checks=False)

plt.figure(figsize=(10, 5))
dend = sch.linkage(condensed_dist, method="ward")
sch.dendrogram(dend, labels=numeric_cols, leaf_rotation=45, color_threshold=0.25)
plt.axhline(y=0.25, color="red", linestyle="--", label="Cutoff (|r|=0.75)")
plt.title("Feature Clustering Dendrogram", fontsize=14)
plt.legend()
plt.tight_layout()
plt.show()


## 4. Outlier Inspection via Grouped Boxplots


In [ ]:
fig, axes = plt.subplots(1, len(numeric_cols), figsize=(16, 5))

for idx, col in enumerate(numeric_cols):
    sns.boxplot(data=telemetry, y=col, x="Machine ID", ax=axes[idx], palette="Set2")
    axes[idx].set_title(f"Boxplot: {col}")

plt.tight_layout()
plt.show()


## 5. Event-Centric Telemetry Degradation Trajectories

Tracking sensor escalation during the 96 hours leading up to known unplanned failures vs false alarms.


In [ ]:
for _, event in events.iterrows():
    m_id = event["Machine ID"]
    e_time = event["Event Timestamp"]
    cat = event["Category"]
    comp = event["Component"] if pd.notna(event["Component"]) else "N/A"
    
    sub = telemetry[(telemetry["Machine ID"] == m_id) & 
                    (telemetry["Timestamp"].between(e_time - pd.Timedelta(hours=96), e_time + pd.Timedelta(hours=24)))].copy()
    
    if sub.empty:
        continue
        
    plt.figure(figsize=(14, 4))
    for col in numeric_cols:
        plt.plot(sub["Timestamp"], sub[col], label=col, alpha=0.8)
    plt.axvline(e_time, color="red" if cat == "Unplanned Failure" else "orange", linestyle="--", linewidth=2, label=f"{cat} ({comp})")
    plt.title(f"Telemetry Trajectory: {m_id} - {cat} ({comp}) at {e_time}", fontsize=13)
    plt.xlabel("Timestamp")
    plt.ylabel("Sensor Value")
    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()
